In [1]:
from opt_targeted_transfers import OptTargetedTransfers
from data_loaders import get_dataset
from data_utils import split_data

In [2]:
# Make train and test sets
X, y, r, features = get_dataset("malawi")
d = 3
(X_train, y_train, r_train), (X_test, y_test, r_test) = split_data(X=X[:, :d], y=y, r=r, p=0.6)

In [3]:
tt = OptTargetedTransfers(name="malawi_example", c_bar=2.15, budget=None)

In [4]:
# Fit density functions
# Note only training for 50 epochs here for an example, in practice use the default number of epochs.
tt.fit(X_train, y_train, r_train, n_epochs=50)

Fitting conditional densities vs glm spline method...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:58<00:00,  1.17s/it, loss=-0.0146, val_loss=0.0177]

Final Theta: tensor([[-0.9873,  0.7758, -0.2372],
        [ 0.5527, -0.3248, -0.0848],
        [-0.3383,  0.4670, -0.5115],
        [-0.1545, -0.1551,  0.1368],
        [-0.5008,  0.4967,  0.3995],
        [-0.0546, -0.3039,  0.3120],
        [ 0.4300, -0.4868,  0.4648],
        [-0.2917, -0.4789, -0.0600]], dtype=torch.float64)


In [5]:
# Set budget (can do this at initialization or after fitting densities)
tt.set_budget(budget=0.1)

In [6]:
# Run optimization algorithm
# Note only trying 10 alpha values for an example, in practice use the default number of alpha values.
opt_policy = tt.run_opt(
    X_test, r_test, n_alpha=10, path="malawi_example_unconditional_budget=0.1.csv"
)

Alpha range: 0.0006727523319816308, 0.6727523319816308


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:18<00:00,  1.84s/it]


In [7]:
# Query the optimal transfer policy after running the optimization algorithm
transfer = opt_policy(X_test[[0]])
print(transfer)

{0: [(1.369919565047933, 1.0)]}


In [8]:
# Note that tt object also stores the optimal policy
transfer = tt.opt_policy(X_test[[0]])
print(transfer)

{0: [(1.369919565047933, 1.0)]}


In [9]:
# Evaluate policy. 
res = tt.evaluate(X_test, y_test, r_test)
res

{'initial_prob_below_line': 0.6511447390812347,
 'initial_poverty_gap': 0.628796805717522,
 'post_transfer_poverty_gap': 0.03889304357416789,
 'post_transfer_prob_below_line': 0.12867077288009346,
 'policy_cost': 1.4717354202223067}

In [10]:
# Can try a different budget without re-fitting the densities!
# Note that setting a new budget will clear the previously computed policy,
# so we will have to re-run the optimization.
tt.set_budget(budget=0.15)
new_opt_policy = tt.run_opt(
    X_test, r_test, n_alpha=10, path="malawi_example_unconditional_budget=0.15.csv"
)

Alpha range: 0.0006727523319816308, 0.6727523319816308


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:17<00:00,  1.71s/it]


In [11]:
# Evaluate policy
res = tt.evaluate(X_test, y_test, r_test)
print(res)

{'initial_prob_below_line': 0.6511447390812347, 'initial_poverty_gap': 0.628796805717522, 'post_transfer_poverty_gap': 0.06718601218786542, 'post_transfer_prob_below_line': 0.16407961731969803, 'policy_cost': 1.3282259185247347}
